In [ ]:
from qmirt.utils.filesystem import find_project_root
import pandas as pd
import numpy as np
import opengate as gate
from pathlib import Path
from qmirt.utils.simulation import (
    make_gate_shell_trd,
    make_gate_shell_box,
    add_pixelated_detector_to_gate_sim
)

In [ ]:
project_root = find_project_root()
print(f"Project root: {project_root}")

Project root: /home/fanghan/Work/RPIL/QMIRT/qmirt-gate-10-sim


In [ ]:
geometry_config_csv_path = project_root / "persistent_data/cardiac_spect/spreadsheet/MDSL.excel80M10RFR.cut-plate.010.150roi.2.30pin.105ellipse_geometry_config.csv"
geometry_config_df = pd.read_csv(geometry_config_csv_path)

In [ ]:
print(geometry_config_df.columns)

Index(['hole_center_x (mm)', 'hole_center_y (mm)', 'hole_center_z (mm)',
       'wall_thickness (mm)', 'hole_r (mm)', 'body_l (mm)',
       'body_l_corrected (mm)', 'body_inner_top (mm)',
       'body_inner_top_corrected (mm)', 'body_outer_top (mm)',
       'body_inner_bottom (mm)', 'body_outer_bottom (mm)', 'guide_l (mm)',
       'guide_inner_top (mm)', 'guide_outer_top (mm)',
       'guide_inner_bottom (mm)', 'guide_outer_bottom (mm)',
       'box_inner_size_x (mm)', 'box_inner_size_y (mm)',
       'box_inner_size_z (mm)', 'box_outer_size_x (mm)',
       'box_outer_size_y (mm)', 'box_outer_size_z (mm)', 'z_axis_angle (deg)',
       'x_axis_angle (deg)'],
      dtype='str')


In [ ]:
print(geometry_config_df[["body_l (mm)","body_l_corrected (mm)","wall_thickness (mm)","body_inner_top (mm)","body_inner_bottom (mm)","body_inner_top_corrected (mm)","body_outer_top (mm)","body_outer_bottom (mm)"]].iloc[0])


body_l (mm)                      89.070206
body_l_corrected (mm)            87.070206
wall_thickness (mm)               2.000000
body_inner_top (mm)              50.000000
body_inner_bottom (mm)            2.300000
body_inner_top_corrected (mm)    48.928935
body_outer_top (mm)              52.928935
body_outer_bottom (mm)            6.300000
Name: 0, dtype: float64


In [ ]:
print(geometry_config_df[["box_outer_size_x (mm)", "box_outer_size_z (mm)"]].iloc[0])
print(geometry_config_df[["box_inner_size_x (mm)", "box_inner_size_z (mm)"]].iloc[0])

box_outer_size_x (mm)    55.2
box_outer_size_z (mm)    16.0
Name: 0, dtype: float64
box_inner_size_x (mm)    51.2
box_inner_size_z (mm)    10.0
Name: 0, dtype: float64


In [ ]:
from scipy.spatial.transform import Rotation


def get_module_rotation_matrix(geometry_config_df: pd.DataFrame, id: int):

    # rotate around x axis by 90 degrees
    rx_0 = Rotation.from_euler("x", -90, degrees=True).as_matrix()
    # Rotate around z axis by 90 degrees
    rz_0 = Rotation.from_euler("z", 90, degrees=True).as_matrix()
    # Then rotate around z axis by the azimuthal angle
    rz_1 = Rotation.from_euler(
        "z", geometry_config_df["z_axis_angle (deg)"][id], degrees=True
    ).as_matrix()
    # Then rotate around y axis by the polar angle
    rx_1 = Rotation.from_euler(
        "x", geometry_config_df["x_axis_angle (deg)"][id], degrees=True
    ).as_matrix()
    r = rz_1 @ rx_1 @ rx_0 @ rz_0
    return r


def create_pyramid_detector_module(
    sim: gate.Simulation,
    *,
    id: int,
    geometry_config_df,
    crystal_size_mm: np.ndarray = np.array([50.0, 50.0, 10.0]),
    pixel_count: np.ndarray = np.array([1, 1, 1]),
):
    body, _ = make_gate_shell_trd(
        sim,
        name="CollimatorBody_" + str(id),
        mother="world",
        top_inner_mm=geometry_config_df["body_inner_top_corrected (mm)"][id],
        top_outer_mm=geometry_config_df["body_outer_top (mm)"][id],
        bottom_inner_mm=geometry_config_df["body_inner_bottom (mm)"][id],
        bottom_outer_mm=geometry_config_df["body_outer_bottom (mm)"][id],
        length_mm=geometry_config_df["body_l_corrected (mm)"][id],
        translation_mm=(geometry_config_df["body_l_corrected (mm)"][id] * 0.5)
        * np.array([0, 0, 1]),
        material="Tungsten",
        inner_material="Air",
    )

    box_shell, box_cavity = make_gate_shell_box(
        sim,
        name="ShieldingBox_" + str(id),
        mother="world",
        outer_size_mm=np.array(
            [
                geometry_config_df["box_outer_size_x (mm)"][id],
                geometry_config_df["box_outer_size_y (mm)"][id],
                geometry_config_df["box_outer_size_z (mm)"][id],
            ]
        ),
        inner_size_mm=np.array(
            [
                geometry_config_df["box_inner_size_x (mm)"][id],
                geometry_config_df["box_inner_size_y (mm)"][id],
                geometry_config_df["box_inner_size_z (mm)"][id],
            ]
        ),
        spacing_size_mm=np.array([0, 0, geometry_config_df["wall_thickness (mm)"][id]]),
        translation_mm=np.array(
            [
                0,
                0,
                geometry_config_df["body_l_corrected (mm)"][id]
                + geometry_config_df["box_outer_size_z (mm)"][id] * 0.5,
            ]
        ),
        inner_shift_mm=np.array(
            [
                0,
                0,
                -(
                    geometry_config_df["box_outer_size_z (mm)"][id]
                    - geometry_config_df["box_inner_size_z (mm)"][id]
                )
                * 0.5
                + geometry_config_df["wall_thickness (mm)"][id],
            ]
        ),
        material="Tungsten",
        inner_material="Air",
    )

    add_pixelated_detector_to_gate_sim(
        sim,
        mother=box_cavity.name,
        id=id,
        translation_mm=np.array(
            [
                0,
                0,
                (crystal_size_mm[2] - geometry_config_df["box_inner_size_z (mm)"][id]),
            ]
        ),
        pixel_count=pixel_count,
        pixel_size_mm=crystal_size_mm / pixel_count,
    )
    frustum_cavity = TrdVolume(
        name=f"box_f_cavity_{id}",
        mother=box_shell.name,
        dx1=geometry_config_df["body_inner_top_corrected (mm)"][id] * 0.5,
        dy1=geometry_config_df["body_inner_top_corrected (mm)"][id] * 0.5,
        dx2=geometry_config_df["body_inner_top (mm)"][id] * 0.5,
        dy2=geometry_config_df["body_inner_top (mm)"][id] * 0.5,
        dz=geometry_config_df["wall_thickness (mm)"][id] * 0.5,
        translation=[
            0,
            0,
            (
                geometry_config_df["wall_thickness (mm)"][id]
                - geometry_config_df["box_outer_size_z (mm)"][id]
            )
            * 0.5,
        ],
        material="Air",
    )

    sim.add_volume(frustum_cavity, name=frustum_cavity.name)

    guide, _ = make_gate_shell_trd(
        sim,
        name="CollimatorGuide_" + str(id),
        mother="world",
        top_inner_mm=geometry_config_df["guide_inner_top (mm)"][id],
        top_outer_mm=geometry_config_df["guide_outer_top (mm)"][id],
        bottom_inner_mm=geometry_config_df["guide_inner_bottom (mm)"][id],
        bottom_outer_mm=geometry_config_df["guide_outer_bottom (mm)"][id],
        length_mm=geometry_config_df["guide_l (mm)"][id],
        translation_mm=-geometry_config_df["guide_l (mm)"][id]
        / 2
        * np.array([0, 0, 1]),
        material="Tungsten",
        inner_material="Air",
    )
    return guide

In [ ]:
sim = gate.Simulation()
sim.volume_manager.add_material_database(
    project_root / "persistent_data" / "GateMaterials.db"
)
sim.user_info.visu = True
sim.user_info.visu_type = "vrml_file_only"
sim.visu_commands_vrml = ["/vis/open VRML2FILE", "/vis/drawVolume"]
sim.visu_commands_vrml.append("/vis/geometry/set/visibility world 0 false")

for module_id in range(10):
    rotation_matrix = get_module_rotation_matrix(geometry_config_df, module_id)
    hole_r = geometry_config_df["hole_r (mm)"].iloc[module_id]
    body_l = geometry_config_df["body_l (mm)"].iloc[module_id]
    guide_l = geometry_config_df["guide_l (mm)"].iloc[module_id]
    translation_mm = (
        geometry_config_df[
            ["hole_center_x (mm)", "hole_center_y (mm)", "hole_center_z (mm)"]
        ]
        .iloc[module_id]
        .to_numpy()
    )
    module_container = create_pyramid_detector_module(
        sim, id=module_id, geometry_config_df=geometry_config_df
    )
    
    module_container.translation = translation_mm
    module_container.rotation=rotation_matrix


visu_filename = "test_cardiac_spect_geometry.wrl"
sim.visu_commands_vrml.append("/vis/viewer/flush")
sim.user_info.visu_filename = str(Path(visu_filename).resolve())
sim.run(start_new_process=True)

Dispatching simulation to subprocess ...
⚠️ No configured source, no particle will be generated.
Simulation: create RunManager (single thread)


Simulation: initialize Geometry
Simulation: initialize Physics
Simulation: initialize Sources
Simulation: initialize Auxiliary attributes
Simulation: initialize Visualization
Simulation: initialize Actors
Simulation: initialize G4RunManager
⚠️ G4Exception origin: G4PVPlacement::CheckOverlaps()
G4Exception code: GeomVol1002
G4Exception severity: G4ExceptionSeverity.JustWarning
G4Exception: Overlap with mother volume !
          Overlap is detected for volume CollimatorBody_0_outer:0 (G4Trd) with its mother volume ModuleContainer_0 (G4Box)
          protrusion at mother local point (17.2304,-5.87895,87.0702) by 3.40351 cm  (max of 690 cases)
NOTE: Reached maximum fixed number -1- of overlaps reports for this volume !
⚠️ G4Exception origin: G4PVPlacement::CheckOverlaps()
G4Exception code: GeomVol1002
G4Exception severity: G4ExceptionSeverity.JustWarning
G4Exception: Overlap with mother volume !
          Overlap is detected for volume ShieldingBox_0_outer:0 (G4Box) with its mother volume 

Process Process-4:
Traceback (most recent call last):
  File "/home/fanghan/.local/share/micromamba/envs/opengate/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/fanghan/.local/share/micromamba/envs/opengate/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/fanghan/.local/share/micromamba/envs/opengate/lib/python3.12/site-packages/opengate/processing.py", line 10, in target_func
    q.put(f(*args, **kwargs))
          ^^^^^^^^^^^^^^^^^^
  File "/home/fanghan/.local/share/micromamba/envs/opengate/lib/python3.12/site-packages/opengate/managers.py", line 2224, in _run_simulation_engine
    output = se.run_engine()
             ^^^^^^^^^^^^^^^
  File "/home/fanghan/.local/share/micromamba/envs/opengate/lib/python3.12/site-packages/opengate/engines.py", line 1612, in run_engine
    self.start_and_stop()
  File "/home/fanghan/.local/share/micromamba/envs/opengate/lib/python

☠️ Fatal in /home/fanghan/.local/share/micromamba/envs/opengate/lib/python3.12/site-packages/opengate/processing.py line 60
☠️ The queue is empty. The spawned process probably died or crashed.


Exception: The queue is empty. The spawned process probably died or crashed.

In [ ]:
from qmirt.plot.wrl import plot_wrl_file

plot_wrl_file("test_cardiac_spect_geometry.wrl")